# BloodMNIST Experiments (CNN or MLP)
Use this notebook as the runner. Shared code lives in the `ml/` package.

In [11]:
from ml.data import build_bloodmnist_dataloaders
from ml.models import CNN, MLP
from ml.setup import get_device, make_criterion, make_optimizer
from ml.training import EarlyStopping, train_with_validation, evaluate_accuracy

In [12]:
BATCH_SIZE = 128
data_bundle = build_bloodmnist_dataloaders(batch_size=BATCH_SIZE, download=True)

train_loader = data_bundle['train_loader']
val_loader = data_bundle['val_loader']
test_loader = data_bundle['test_loader']
n_channels = data_bundle['n_channels']
n_classes = data_bundle['n_classes']

print(f'n_channels={n_channels}, n_classes={n_classes}')

n_channels=3, n_classes=8


In [13]:
MODEL_NAME = 'mlp'   # choose: 'cnn' or 'mlp'
OPTIMIZER_NAME = 'sgd'  # choose: 'adam' or 'sgd'
LEARNING_RATE = 0.001 # 0.01 for sgd
MOMENTUM = 0.9
DROPOUT = 0.0
PATIENCE = 50
EARLY_STOPIING_DELTA = 0.0002
WEIGHT_DECAY = 0.0005

device = get_device()

if MODEL_NAME == 'cnn':
    model = CNN(in_channels=n_channels, num_classes=n_classes, dropout=DROPOUT).to(device)
elif MODEL_NAME == 'mlp':
    model = MLP(input_dim=n_channels * 28 * 28, num_classes=n_classes, dropout=DROPOUT).to(device)
else:
    raise ValueError('MODEL_NAME must be cnn or mlp')

criterion = make_criterion('cross_entropy')
optimizer = make_optimizer(
    model,
    name=OPTIMIZER_NAME,
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
early_stopping = EarlyStopping(PATIENCE, EARLY_STOPIING_DELTA)

print(f'Device: {device}')
print(f'Model: {MODEL_NAME}')
print(f'Optimizer: {OPTIMIZER_NAME}, lr={LEARNING_RATE}')

Device: cpu
Model: mlp
Optimizer: sgd, lr=0.001


In [14]:
history = train_with_validation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=500,
    early_stopping=early_stopping,
)

print('Stopped epoch:', history['stopped_epoch'])
print('Best epoch:', history['best_epoch'])

Epoch 1	Training Loss: 1.9871	Validation Loss: 1.9434
Epoch 10	Training Loss: 1.0771	Validation Loss: 1.0332
Epoch 20	Training Loss: 0.8323	Validation Loss: 0.7894
Epoch 30	Training Loss: 0.7238	Validation Loss: 0.6856
Epoch 40	Training Loss: 0.6303	Validation Loss: 0.6167
Epoch 50	Training Loss: 0.5786	Validation Loss: 0.5544
Epoch 60	Training Loss: 0.5412	Validation Loss: 0.5147
Epoch 70	Training Loss: 0.5072	Validation Loss: 0.5029
Epoch 80	Training Loss: 0.5069	Validation Loss: 0.4706
Epoch 90	Training Loss: 0.4673	Validation Loss: 0.4759
Epoch 100	Training Loss: 0.4513	Validation Loss: 0.4940
Epoch 110	Training Loss: 0.4301	Validation Loss: 0.4800
Epoch 120	Training Loss: 0.4045	Validation Loss: 0.4154
Epoch 130	Training Loss: 0.3891	Validation Loss: 0.4810
Epoch 140	Training Loss: 0.3899	Validation Loss: 0.4446
Epoch 150	Training Loss: 0.3756	Validation Loss: 0.4589
Epoch 160	Training Loss: 0.3547	Validation Loss: 0.3992
Epoch 170	Training Loss: 0.3445	Validation Loss: 0.4953
Epo

In [15]:
test_accuracy = evaluate_accuracy(model, test_loader, device)
print(f'Test accuracy: {test_accuracy:.2f}%')

Test accuracy: 86.70%
